## Importing Libraries

In [30]:
import textwrap
import numpy as np
import pandas as pd

from typing import List

import google.generativeai as genai
import google.ai.generativelanguage as glm

from PyPDF2 import PdfReader

# Importing the CharacterTextSplitter class from the langchain library to split the text into chunks
from langchain.text_splitter import CharacterTextSplitter

from pinecone import Pinecone


from IPython.display import Markdown

import getpass
import os


## Google Gemini (LLM) model Configuration

In [31]:
GOOGLE_API_KEY=getpass.getpass()
genai.configure(api_key=GOOGLE_API_KEY)

## Pinecone (Vector DB) Configuration

In [32]:
pc = Pinecone("492fe419-7850-4384-8dc4-d2019c9d1ab2")

In [33]:
# Extract the content of the PDF
pdf_content = ""
# Loop through the PDF files
pdf_docs = ["CV.pdf"]
for pdf in pdf_docs:
    # Read the PDF file
    pdf_reader = PdfReader(pdf)
    # Loop through the pages of the PDF file
    for page in pdf_reader.pages:
        # Extract the text from the PDF page and add it to the pdf_content variable
        pdf_content += page.extract_text()
# st.write(pdf_content)
# Get chunks of the content
# Split the text into chunks of 2000 characters with an overlap of 200 characters
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=2000,
    chunk_overlap=200,
    length_function=len,
)
# Split the text into chunks of 2000 characters with an overlap of 200 characters
chunks = text_splitter.split_text(pdf_content)

In [34]:
chunks[0]

'Amr  Ghribi\namrou.ghribi@esprit.tn  | (+49) 15510913348 | Schmalkalden, Germany | linkedin.com/in/amrou-ghribi\nEDUCA TION\nHochschule Schmalkalden Schmalkalden, Thuringen, Germany\nMasters in Computer Science\nESPRIT Tunis, Tunisia\nComputer Science Engineering specialized in Data Science Graduation Date: Sep 2024\nWORK EXPERIENCE\nValue - AI For  Capital Markets Tunis\nInternship  - Jun 2024 Jul 2024\nDeveloped an automated system for Citi Bank, focusing on Alstom, to generate one-year forecast research reports on \nstocks, reducing report generation time to one day .\nUtilized Google’ s LLM Gemini , Retrieval-Augmented Generation (RAG) , and Postgr eSQL  for database \nmanagement, with data science techniques to structure and clean unstructured financial data.\nSystem was successfully sold to a client, demonstrating its commercial viability .\nBourse de Tunis - Tunis Stock Exchange Tunis\nInternship  - Jul 2023 Aug 2023\nGained hands-on experience in finance and technology integra

In [35]:
df = pd.DataFrame(chunks)
df.columns = ["Content"]
df

,Content
0,Amr Ghribi\namrou.ghribi@esprit.tn | (+49) 1...
1,precision.\nDeployed on Vercel with minimal la...


In [36]:


embeddings = pc.inference.embed(
    "multilingual-e5-large",
    inputs=df['Content'].tolist(),
    parameters={
        "input_type": "passage"
    }
)

vectors = []
for i, (d, e) in enumerate(zip(df['Content'], embeddings)):
    vectors.append({
        "id": str(i),
        "values": e['values'],
        "metadata": {'text': d}
    })

index = pc.Index('store')

index.upsert(
    vectors=vectors,
    namespace="ns1"
)

{'upserted_count': 2}

## Check which Gemini models are available for use

In [37]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-1.0-pro-latest
models/gemini-1.0-pro
models/gemini-pro
models/gemini-1.0-pro-001
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-001
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-pro-exp-0801
models/gemini-1.5-pro-exp-0827
models/gemini-1.5-flash-latest
models/gemini-1.5-flash-001
models/gemini-1.5-flash-001-tuning
models/gemini-1.5-flash
models/gemini-1.5-flash-exp-0827
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-1.5-flash-8b-exp-0827
models/gemini-1.5-flash-8b-exp-0924


## We'll be using Gemini 1.5-flash

In [38]:
model = genai.GenerativeModel('models/gemini-1.5-flash')

## Build the prompt for the LLM

In [43]:
def build_prompt(query: str, context: List[str]) -> str:
    """
    Builds a prompt for the LLM. #

    This function builds a prompt for the LLM. It takes the original query,
    and the returned context, and asks the model to answer the question based only
    on what's in the context, not what's in its weights.

    Args:
    query (str): The original query.
    context (List[str]): The context of the query, returned by embedding search.

    Returns:
    A prompt for the LLM (str).
    """

    base_prompt = {
        "content": "You are a human resources expert working for a renewed IT company. You are tasked with checking the CVs given to you and request thing to change and show the good things in the CVs. Answer only based on the context provided. Do not explain your answer.",
    }
    user_prompt = {
        "content": f" The question is '{query}'. Here is all the context you have:"
        f'{(" ").join(context)}',
    }

    # combine the prompts to output a single prompt string
    system = f"{base_prompt['content']} {user_prompt['content']}"

    return system


## Generating Gemini response

In [44]:
def get_gemini_response(query: str, context: List[str]) -> str:
    """
    Queries the Gemini API to get a response to the question.

    Args:
    query (str): The original query.
    context (List[str]): The context of the query, returned by embedding search.

    Returns:
    A response to the question.
    """

    response = model.generate_content(build_prompt(query, context))

    return response.text

## Chatting with the LLM

In [47]:
import streamlit as st


st.title("Chat with Gemini")

# Initialize session state for query and response
if "query" not in st.session_state:
    st.session_state.query = ""
if "response" not in st.session_state:
    st.session_state.response = ""

# Input box for user query
query = st.text_input("Query:", value=st.session_state.query)

if st.button("Submit"):
    if len(query) == 0:
        st.write("Please enter a question.")
    else:
        st.session_state.query = query
        st.write("Thinking...")

        x = pc.inference.embed(
            model="multilingual-e5-large",
            inputs=[query],
            parameters={
                "input_type": "query"
            }
        )

        results = index.query(
            namespace="ns1",
            vector=x[0].values,
            top_k=3,
            include_values=False,
            include_metadata=True
        )

        context = [match['metadata']['text'] for match in results['matches']]
        response = get_gemini_response(query, context)

        st.session_state.response = response

# Display the response
if st.session_state.response:
    st.write(f"Question: {st.session_state.query}")
    st.write(f"Response: {st.session_state.response}")

2024-11-11 20:22:12.884 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-11-11 20:22:12.918 
  command:

    streamlit run /Users/user/Library/Python/3.12/lib/python/site-packages/ipykernel_launcher.py [ARGUMENTS]
2024-11-11 20:22:12.919 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-11-11 20:22:12.920 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-11-11 20:22:12.920 Session state does not function when running a script without `streamlit run`
2024-11-11 20:22:12.921 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-11-11 20:22:12.922 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-11-11 20:22:12.922 Thread 'MainThread': missing Scri